In [3]:
!pip -q install datasets pandas scikit-learn gradio joblib

In [4]:
import pandas
import sklearn
import gradio
import datasets
import joblib

print("All libraries installed successfully!")

All libraries installed successfully!


In [5]:
from datasets import load_dataset

print("Downloading TrustLens AI dataset...")

dataset = load_dataset(
    "David-Egea/phishing-texts",
    split="train"
)

df = dataset.to_pandas()

print("Dataset downloaded successfully!")
print("Dataset shape:", df.shape)
print("Dataset columns:", df.columns.tolist())

display(df.head(10))

README.md:   0%|          | 0.00/1.90k [00:00<?, ?B/s]

dataset.json: reconstructing file:   0%|          |  0.00B / 52.3MB            

dataset.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/20301 [00:00<?, ? examples/s]

Dataset downloaded successfully!
Dataset shape: (20301, 2)
Dataset columns: ['phishing', 'text']


,phishing,text
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ..."
1,0,the other side of * galicismos * * galicismo *...
2,0,re : equistar deal tickets are you still avail...
3,1,\nHello I am your hot lil horny toy.\n I am...
4,1,software at incredibly low prices ( 86 % lower...
5,0,global risk management operations sally congra...
6,0,"On Sun, Aug 11, 2002 at 11:17:47AM +0100, wint..."
7,1,"entourage , stockmogul newsletter ralph velez ..."
8,1,"we owe you lots of money dear applicant , afte..."
9,0,re : coastal deal - with exxon participation u...


In [6]:
from datasets import load_dataset

print("Downloading TrustLens AI dataset...")

dataset = load_dataset(
    "David-Egea/phishing-texts",
    split="train"
)

df = dataset.to_pandas()

print("\nDataset downloaded successfully!")
print("Total records:", len(df))
print("Dataset columns:", df.columns.tolist())

display(df.head(10))


Dataset downloaded successfully!
Total records: 20301
Dataset columns: ['phishing', 'text']


,phishing,text
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ..."
1,0,the other side of * galicismos * * galicismo *...
2,0,re : equistar deal tickets are you still avail...
3,1,\nHello I am your hot lil horny toy.\n I am...
4,1,software at incredibly low prices ( 86 % lower...
5,0,global risk management operations sally congra...
6,0,"On Sun, Aug 11, 2002 at 11:17:47AM +0100, wint..."
7,1,"entourage , stockmogul newsletter ralph velez ..."
8,1,"we owe you lots of money dear applicant , afte..."
9,0,re : coastal deal - with exxon participation u...


In [7]:

import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# 1. Clean the dataset
df = df[["text", "phishing"]].copy()

df = df.dropna()
df["text"] = df["text"].astype(str).str.strip()
df["phishing"] = pd.to_numeric(
    df["phishing"], errors="coerce"
)

df = df.dropna()
df["phishing"] = df["phishing"].astype(int)
df = df[df["phishing"].isin([0, 1])]
df = df[df["text"].str.len() > 10]
df = df.drop_duplicates(subset=["text"])

print("Clean dataset size:", len(df))

print("\nClass distribution:")
print(df["phishing"].value_counts())

# 2. Separate text and labels
X = df["text"]
y = df["phishing"]

# 3. Split data: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining records:", len(X_train))
print("Testing records:", len(X_test))

# 4. Create the machine learning pipeline
model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            max_features=30000,
            ngram_range=(1, 2),
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced"
        )
    )
])

# 5. Train the AI model
print("\nTraining TrustLens AI...")

model.fit(X_train, y_train)

print("MODEL TRAINING COMPLETED!")

# 6. Test the model
predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)
precision = precision_score(
    y_test, predictions, zero_division=0
)
recall = recall_score(
    y_test, predictions, zero_division=0
)
f1 = f1_score(
    y_test, predictions, zero_division=0
)

# 7. Display evaluation results
print("\n========== TRUSTLENS AI RESULTS ==========")

print(f"Accuracy:  {accuracy * 100:.2f}%")
print(f"Precision: {precision * 100:.2f}%")
print(f"Recall:    {recall * 100:.2f}%")
print(f"F1 Score:  {f1 * 100:.2f}%")

print("\nDetailed Classification Report:")

print(classification_report(
    y_test,
    predictions,
    target_names=["Non-phishing", "Phishing"],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))

# 8. Save the trained model
joblib.dump(model, "trustlens_model.pkl")

print("\nTrustLens AI model saved successfully!")


Clean dataset size: 20155

Class distribution:
phishing
0    12576
1     7579
Name: count, dtype: int64

Training records: 16124
Testing records: 4031

Training TrustLens AI...
MODEL TRAINING COMPLETED!

========== TRUSTLENS AI RESULTS ==========
Accuracy:  96.55%
Precision: 94.62%
Recall:    96.31%
F1 Score:  95.46%

Detailed Classification Report:
              precision    recall  f1-score   support

Non-phishing       0.98      0.97      0.97      2515
    Phishing       0.95      0.96      0.95      1516

    accuracy                           0.97      4031
   macro avg       0.96      0.97      0.96      4031
weighted avg       0.97      0.97      0.97      4031

Confusion Matrix:
[[2432   83]
 [  56 1460]]

TrustLens AI model saved successfully!


In [8]:

import gradio as gr
import re

def analyze_message(message):

    if not message or len(message.strip()) < 10:
        return (
            "Please enter a message containing at least 10 characters.",
            "",
            ""
        )

    # Get prediction from our trained model
    probabilities = model.predict_proba([message])[0]
    phishing_index = list(model.classes_).index(1)
    score = probabilities[phishing_index] * 100

    # Interpret the model score
    if score >= 70:
        result = "HIGH PHISHING RISK"
        explanation = (
            "The model identifies strong similarities "
            "to phishing messages in its training data."
        )
    elif score >= 40:
        result = "REQUIRES FURTHER REVIEW"
        explanation = (
            "The model found some phishing-like patterns. "
            "Verify the sender and website independently."
        )
    else:
        result = "LOWER MODEL-DETECTED RISK"
        explanation = (
            "The model found fewer phishing-like patterns. "
            "This does not guarantee that the message is safe."
        )

    # Identify possible warning signs
    warning_patterns = {
        "Urgent language": r"\b(urgent|immediately|act now|account suspended)\b",
        "Password or verification request": r"\b(password|otp|verification code|pin)\b",
        "Unusual payment request": r"\b(send money|transfer fee|processing fee|advance payment)\b",
        "Unexpected prize": r"\b(you have won|claim your prize|winner|free gift)\b",
        "Web link included": r"https?://|www\."
    }

    warnings = []

    for name, pattern in warning_patterns.items():
        if re.search(pattern, message, re.IGNORECASE):
            warnings.append("• " + name)

    if not warnings:
        warnings.append(
            "• No warning phrases detected by the basic rule checker."
        )

    result_text = result + "\n\n" + explanation
    score_text = f"Model phishing score: {score:.2f}%"
    warning_text = "\n".join(warnings)

    return result_text, score_text, warning_text


# Professional interface
with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown(
        """
        # TRUSTLENS AI
        ### Intelligent Phishing Message Detection

        Analyze suspicious emails, SMS messages, and online offers
        using a trained machine-learning model.
        """
    )

    message_input = gr.Textbox(
        label="Enter your message",
        placeholder="Paste a suspicious message here...",
        lines=7
    )

    analyze_button = gr.Button(
        "Analyze Message",
        variant="primary"
    )

    result_output = gr.Textbox(
        label="Detection Result",
        lines=3
    )

    score_output = gr.Textbox(
        label="Model Score"
    )

    warnings_output = gr.Textbox(
        label="Possible Warning Signs",
        lines=5
    )

    analyze_button.click(
        fn=analyze_message,
        inputs=message_input,
        outputs=[
            result_output,
            score_output,
            warnings_output
        ]
    )

    gr.Markdown(
        """
        ---
        **Disclaimer:** TrustLens AI is an educational research
        prototype. Its predictions are not proof of fraud.
        Always verify important messages through official sources.
        """
    )

demo.launch(share=True)


/tmp/ipykernel_544/1884911627.py:66: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://189b823cf51a209fb4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Congratulations! You have won a free prize.
Click this link immediately and provide your password
and verification code to claim your reward.